# 7. Leaderboard and hypotheses

> **Runtime: CPU.** Runtime → Change runtime type → **CPU**. Statistics on the tables of notebook 6.

**Overall leaderboard** of every method by fidelity and utility; privacy and cost are not part of it (`marathonts/hypotheses/overall.py`).

* **Fidelity** — the core metric set, one or two canonical metrics per criterion fixed in advance (distribution, dependence, windows, long structure, target, seams); each criterion is the mean of its ranks against **train, val and OOT** with equal weights.
* **Utility** — TSTR, classification and regression with equal weights, each tested on val and OOT.
* **Overall** = the mean of the two domains with equal weights.
* **Noise-aware ranks**: within a leaf two methods whose scores differ by no more than the noise of the leaf (median over seeds of |random − twin|, or the between-seed spread of the random path where the twin is not run, as in `min`) tie; with zero noise these are the ordinary average ranks.
* Next to the rank: fidelity and utility ranks alone and their Pareto front, normalised scores (random assembly = 0, real data = 1), a memorisation flag (near copies of train against the real OOT piece; it does not change the rank).
* Robustness: sensitivity variants (plain ranks, train only, val + OOT only, without the metrics that have no anchor check, every metric, common datasets), one method per generator, leave-one-dataset-out, random criterion weights.

The same run also writes the **detailed ranking** of every method in every slice (the four rankings, core-metric rankings, privacy, cost, criteria, themes, stability — `LEADERBOARD.md`), the **metric values** (values of the core fidelity metrics and the primary utility metrics by reference, dataset, task, model and split — `METRIC_VALUES.md`) and the **report** with research questions RQ0–RQ9, the confirmatory family C1–C4 and the case study of one method CS1–CS15 (`REPORT.md`, HYPOTHESES.md).

Output → reports `artifacts/<profile>/reports/` (`OVERALL.md`, `LEADERBOARD.md`, `METRIC_VALUES.md`, `REPORT.md`); tables `artifacts/<profile>/overall/`, `leaderboard/`, `metric_values/`, `hypotheses/`.


In [ ]:
# ── Drive first: the project (and colab_setup.py) lives there ──
from google.colab import drive
drive.mount("/content/drive")

# ── setup: Drive, code, dependencies, paths ──
REPO = "/content/drive/MyDrive/Colab Notebooks/MarathonTS"   # project folder on Drive (or a git clone)
PROFILE = "min"            # "min": the Colab run (fast); "mid": min + the VAE per seed; "max": everything on
# results go to {REPO}/artifacts/<PROFILE>; set ARTIFACTS = "..." to change
exec(open(f"{REPO}/colab/colab_setup.py").read())

In [ ]:
import sys
from marathonts.hypotheses.run import main
import tempfile
from pathlib import Path
cfg_json = Path(tempfile.gettempdir()) / "analysis_config.json"   # this notebook's cfg, analysis settings included
cfg_json.write_text(cfg.to_json())
main(["--config", str(cfg_json), "--artifacts-dir", str(cfg.artifacts_dir)])          # add "--plugins", "my_tests.py" for custom tests
# reports → artifacts/<profile>/reports/; "--out-dir", "X" — base folder: X/reports, X/hypotheses, X/overall, …
# rankings leave out the seam-processing variants (min, mid; Config.rank_exclude); add
# "--rank-exclude" with no names to rank every method, or with names to choose the list.
# Analysis-only choices (nothing is recomputed, docs/CONFIGURATION.md, "Analysis-only settings"):
#   "--exclude-datasets", "bitcoin"                 datasets left out of the analysis
#   "--exclude-metrics", "F6.1_disc_score@val"      metrics left out (id, id@ref, group prefix "F12.", utility metric)
#   "--leaked", '{"chronos": ["electricity"]}'      replaces the leakage list (default: marathonts/bench/config.py LEAKED)


In [ ]:
import pandas as pd
from IPython.display import Markdown, display
from marathonts.hypotheses.leaderboard import column_doc
from marathonts.hypotheses.overall import doc

ov = cfg.artifacts_dir / "overall"
if not (ov / "overall.parquet").exists():   # the overall step failed: see its message in the cell above
    raise FileNotFoundError(f"no overall leaderboard in {ov}: see the FAILED line of the run above")
read = lambda name, idx="method": pd.read_parquet(ov / f"{name}.parquet").set_index(idx)
m = read("overall")
cols = ["family", "avg_rank", "ci_lo", "ci_hi", "p_first", "p_top3", "p_worse_than_best_holm", "tier_wilcoxon", "n_datasets"]
display(Markdown(f"### Overall leaderboard\n\nRows sorted by avg_rank (1 = best).\n\n{column_doc(m[cols])}"))
display(m[cols].round(2))


In [ ]:
side = [c for c in ("fidelity_rank", "utility_rank", "pareto_fid_ut", "norm_fidelity", "norm_utility", "norm_overall",
                    "near_copy_share", "near_copy_real", "near_copy_ratio", "p_more_copies_holm", "memorisation_flag") if c in m]
display(Markdown(f"### Next to the rank\n\n{doc(m[side])}"))
display(m[side].round(2))
display(Markdown("### Criteria\n\nMean rank of every method in each criterion (references / test splits averaged)."))
display(read("criteria").round(2))


In [ ]:
v = read("variants", "variant")
display(Markdown(f"### Sensitivity\n\n{doc(v)}"))
display(v.round(2))
display(Markdown("avg_rank of every method in every variant"))
display(read("sensitivity").round(2))


In [ ]:
f, lo, w = read("field"), read("lodo"), read("weights")
display(Markdown(f"### One method per generator\n\n{doc(f)}"))
display(f.round(2))
display(Markdown(f"### Leave one dataset out\n\n{doc(lo)}"))
display(lo)
display(Markdown("### Random criterion weights\n\nDirichlet(1) within each domain, 2000 draws (median Kendall τ with equal "
                 "weights — OVERALL.md)."))
display(w.round(2))
rep = cfg.artifacts_dir / "reports"
print(f"overall: {rep / 'OVERALL.md'}\ndetailed ranking: {rep / 'LEADERBOARD.md'}\n"
      f"metric values: {rep / 'METRIC_VALUES.md'}\nreport: {rep / 'REPORT.md'}")
